# 25. Query Decomposition RAG (NVIDIA Pattern)

## Overview
Query Decomposition breaks complex queries into sub-questions, retrieving information for each before synthesizing the final answer.

**Pattern Source**: NVIDIA RAG Factory  
**Use Case**: Multi-hop reasoning, queries spanning multiple documents

## Architecture
```
Complex Query → Decompose → Sub-Questions
                              ↓
                  ┌───────────┼───────────┐
                  ↓           ↓           ↓
              RAG Tool    RAG Tool    Math Tool
                  ↓           ↓           ↓
              Answer 1    Answer 2    Result
                  └───────────┼───────────┘
                              ↓
                    Synthesize Final Answer
```

## Key Innovation
- Recursive decomposition until all sub-questions answered
- Custom agent with multiple tools (search, math, synthesis)
- Parallel sub-question processing when possible
- Tracks dependencies between sub-questions

## When to Use
- Complex analytical questions
- Queries requiring information from multiple documents
- Comparison queries ("what's the difference between X and Y?")
- Computational queries on retrieved data

In [ ]:
import boto3
import json
from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import re
from typing import List, Dict, Tuple
import time

## Configuration

In [ ]:
# AWS Configuration
region = 'us-east-1'
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region)

# OpenSearch Configuration
host = 'YOUR_OPENSEARCH_ENDPOINT'
index_name = 'query-decomposition-rag'

# Model IDs
CLAUDE_MODEL_ID = 'anthropic.claude-sonnet-4-20250514-v1:0'
EMBED_MODEL_ID = 'amazon.titan-embed-text-v2:0'

# OpenSearch client setup
service = 'aoss'
credentials = boto3.Session().get_credentials()
awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    service,
    session_token=credentials.token
)

os_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)

## Sample Knowledge Base Setup

In [ ]:
# Sample documents for testing
documents = [
    {
        'id': 'doc1',
        'title': 'AWS Bedrock Pricing',
        'content': """AWS Bedrock Claude Sonnet 4 pricing:
        - Input: $3.00 per million tokens
        - Output: $15.00 per million tokens
        Suitable for complex reasoning and analysis tasks."""
    },
    {
        'id': 'doc2',
        'title': 'AWS Bedrock Haiku Pricing',
        'content': """AWS Bedrock Claude Haiku 4 pricing:
        - Input: $0.80 per million tokens
        - Output: $4.00 per million tokens
        Optimized for speed and efficiency in simple tasks."""
    },
    {
        'id': 'doc3',
        'title': 'Model Capabilities Comparison',
        'content': """Model capabilities:
        - Claude Sonnet 4: Best for complex analysis, coding, research. Context window: 200K tokens.
        - Claude Haiku 4: Best for classification, simple Q&A, data extraction. Context window: 200K tokens.
        - Sonnet has higher accuracy but Haiku is 4x faster."""
    },
    {
        'id': 'doc4',
        'title': 'RAG Cost Optimization',
        'content': """For RAG systems, typical token usage per query:
        - Simple retrieval: 2,000 input tokens (context) + 500 output tokens
        - Complex multi-hop: 5,000 input tokens + 1,000 output tokens
        Cost optimization: Use smaller models for retrieval, larger for complex reasoning."""
    }
]

def get_embedding(text: str) -> List[float]:
    """Get embedding from Titan."""
    body = json.dumps({'inputText': text[:8000]})
    response = bedrock_runtime.invoke_model(modelId=EMBED_MODEL_ID, body=body)
    return json.loads(response['body'].read())['embedding']

def setup_knowledge_base():
    """Create index and load documents."""
    # Create index
    if not os_client.indices.exists(index=index_name):
        index_body = {
            'settings': {
                'index': {'knn': True, 'number_of_shards': 1, 'number_of_replicas': 0}
            },
            'mappings': {
                'properties': {
                    'id': {'type': 'keyword'},
                    'title': {'type': 'text'},
                    'content': {'type': 'text'},
                    'embedding': {
                        'type': 'knn_vector',
                        'dimension': 1024,
                        'method': {'name': 'hnsw', 'space_type': 'cosinesimil', 'engine': 'faiss'}
                    }
                }
            }
        }
        os_client.indices.create(index=index_name, body=index_body)
        print(f"Created index: {index_name}")
    
    # Index documents
    for doc in documents:
        embedding = get_embedding(f"{doc['title']} {doc['content']}")
        os_client.index(
            index=index_name,
            body={'id': doc['id'], 'title': doc['title'], 'content': doc['content'], 'embedding': embedding},
            id=doc['id'],
            refresh=True
        )
        print(f"Indexed: {doc['title']}")
        time.sleep(0.3)

setup_knowledge_base()
print("\nKnowledge base ready!")

## Step 1: Query Decomposition Agent

In [ ]:
def decompose_query(query: str) -> List[str]:
    """
    Decompose complex query into sub-questions using Claude.
    """
    prompt = f"""You are a query decomposition expert. Break down this complex question into simpler sub-questions that need to be answered to fully address the original question.

Rules:
1. Each sub-question should be self-contained and answerable independently
2. Sub-questions should cover all aspects needed to answer the original question
3. Order sub-questions logically (foundational questions first)
4. Return ONLY the sub-questions, one per line, numbered

Original Question: {query}

Sub-questions:"""
    
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 500,
        "temperature": 0.3,
        "messages": [{"role": "user", "content": prompt}]
    })
    
    response = bedrock_runtime.invoke_model(modelId=CLAUDE_MODEL_ID, body=body)
    response_body = json.loads(response['body'].read())
    text = response_body['content'][0]['text']
    
    # Parse numbered sub-questions
    sub_questions = []
    for line in text.strip().split('\n'):
        line = line.strip()
        if line and (line[0].isdigit() or line.startswith('-')):
            # Remove numbering
            question = re.sub(r'^[\d\-\.\)\s]+', '', line).strip()
            if question:
                sub_questions.append(question)
    
    return sub_questions

# Test decomposition
complex_query = "What's the cost difference between using Claude Sonnet 4 vs Haiku 4 for processing 1 million tokens in a RAG system, and which model is more suitable for complex analysis?"

sub_questions = decompose_query(complex_query)

print("Original Query:")
print(f"  {complex_query}\n")
print("Decomposed Sub-Questions:")
for idx, sq in enumerate(sub_questions, 1):
    print(f"  {idx}. {sq}")

## Step 2: RAG Tool (Retrieve & Answer)

In [ ]:
def rag_search_tool(question: str, top_k: int = 3) -> Dict:
    """
    RAG tool: Retrieve relevant documents and generate answer.
    """
    # Retrieve documents
    query_embedding = get_embedding(question)
    
    search_query = {
        'size': top_k,
        'query': {
            'knn': {
                'embedding': {'vector': query_embedding, 'k': top_k}
            }
        },
        '_source': ['title', 'content']
    }
    
    results = os_client.search(index=index_name, body=search_query)
    hits = results['hits']['hits']
    
    if not hits:
        return {'answer': "No relevant information found.", 'sources': []}
    
    # Build context
    context_parts = []
    sources = []
    
    for hit in hits:
        source = hit['_source']
        context_parts.append(f"[{source['title']}]\n{source['content']}")
        sources.append({'title': source['title'], 'score': hit['_score']})
    
    context = "\n\n".join(context_parts)
    
    # Generate answer
    prompt = f"""Based on the following context, answer the question concisely and accurately.
If the answer is not in the context, say "Information not available".

Context:
{context}

Question: {question}

Answer:"""
    
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 300,
        "temperature": 0.5,
        "messages": [{"role": "user", "content": prompt}]
    })
    
    response = bedrock_runtime.invoke_model(modelId=CLAUDE_MODEL_ID, body=body)
    response_body = json.loads(response['body'].read())
    answer = response_body['content'][0]['text']
    
    return {'answer': answer, 'sources': sources}

# Test RAG tool
test_result = rag_search_tool("What is the pricing for Claude Sonnet 4?")
print("RAG Tool Test:")
print(f"Answer: {test_result['answer']}")
print(f"Sources: {[s['title'] for s in test_result['sources']]}")

## Step 3: Math Tool (For Computational Queries)

In [ ]:
def math_tool(expression: str, context: str = "") -> Dict:
    """
    Math tool: Extract numbers from context and perform computation using Claude.
    """
    prompt = f"""You are a mathematical computation assistant. Given the context and the calculation request, perform the computation and return ONLY the numerical result with brief explanation.

Context:
{context}

Computation Request: {expression}

Provide the answer in this exact format:
Result: [numerical value]
Explanation: [one sentence]

Answer:"""
    
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 200,
        "temperature": 0.1,
        "messages": [{"role": "user", "content": prompt}]
    })
    
    response = bedrock_runtime.invoke_model(modelId=CLAUDE_MODEL_ID, body=body)
    response_body = json.loads(response['body'].read())
    answer = response_body['content'][0]['text']
    
    return {'answer': answer, 'type': 'computation'}

# Test math tool
test_math = math_tool(
    "Calculate the cost for 1 million tokens",
    "Claude Sonnet 4 costs $3 per million input tokens and $15 per million output tokens"
)
print("\nMath Tool Test:")
print(f"Answer: {test_math['answer']}")

## Step 4: Query Decomposition Agent (Full Pipeline)

In [ ]:
class QueryDecompositionAgent:
    """
    Agent that decomposes complex queries and orchestrates sub-question answering.
    """
    def __init__(self):
        self.tools = {
            'search': rag_search_tool,
            'math': math_tool
        }
    
    def select_tool(self, question: str) -> str:
        """Determine which tool to use for a sub-question."""
        # Simple heuristic: if question contains math keywords, use math tool
        math_keywords = ['calculate', 'compute', 'cost', 'difference', 'sum', 'total', 'compare numerically']
        
        question_lower = question.lower()
        if any(keyword in question_lower for keyword in math_keywords):
            return 'math'
        return 'search'
    
    def answer_sub_question(self, sub_question: str, accumulated_context: str = "") -> Dict:
        """Answer a single sub-question using the appropriate tool."""
        tool_name = self.select_tool(sub_question)
        
        if tool_name == 'search':
            result = self.tools['search'](sub_question)
            return {
                'question': sub_question,
                'tool': 'search',
                'answer': result['answer'],
                'sources': result.get('sources', [])
            }
        elif tool_name == 'math':
            result = self.tools['math'](sub_question, accumulated_context)
            return {
                'question': sub_question,
                'tool': 'math',
                'answer': result['answer'],
                'sources': []
            }
    
    def synthesize_final_answer(self, original_query: str, sub_answers: List[Dict]) -> str:
        """Synthesize final answer from sub-question answers."""
        # Build context from sub-answers
        context_parts = []
        for sa in sub_answers:
            context_parts.append(f"Q: {sa['question']}\nA: {sa['answer']}")
        
        context = "\n\n".join(context_parts)
        
        prompt = f"""Based on the answers to the sub-questions below, provide a comprehensive answer to the original question.

Sub-Question Answers:
{context}

Original Question: {original_query}

Comprehensive Answer:"""
        
        body = json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 600,
            "temperature": 0.7,
            "messages": [{"role": "user", "content": prompt}]
        })
        
        response = bedrock_runtime.invoke_model(modelId=CLAUDE_MODEL_ID, body=body)
        response_body = json.loads(response['body'].read())
        return response_body['content'][0]['text']
    
    def process_query(self, query: str) -> Dict:
        """Full query decomposition pipeline."""
        print(f"Processing query: {query}\n")
        
        # Step 1: Decompose query
        print("Step 1: Decomposing query...")
        sub_questions = decompose_query(query)
        print(f"Decomposed into {len(sub_questions)} sub-questions\n")
        
        # Step 2: Answer each sub-question
        print("Step 2: Answering sub-questions...")
        sub_answers = []
        accumulated_context = ""
        
        for idx, sq in enumerate(sub_questions, 1):
            print(f"  {idx}. {sq}")
            result = self.answer_sub_question(sq, accumulated_context)
            sub_answers.append(result)
            print(f"     Tool: {result['tool']}")
            print(f"     Answer: {result['answer'][:100]}...\n")
            
            # Accumulate context for dependent questions
            accumulated_context += f"\nQ: {sq}\nA: {result['answer']}\n"
        
        # Step 3: Synthesize final answer
        print("Step 3: Synthesizing final answer...")
        final_answer = self.synthesize_final_answer(query, sub_answers)
        
        return {
            'query': query,
            'sub_questions': sub_questions,
            'sub_answers': sub_answers,
            'final_answer': final_answer
        }

# Create agent
agent = QueryDecompositionAgent()

# Test with complex query
result = agent.process_query(complex_query)

print("\n" + "="*80)
print("FINAL RESULT")
print("="*80)
print(f"\nOriginal Query:\n{result['query']}\n")
print(f"Final Answer:\n{result['final_answer']}")

## Test Cases: Multi-Hop Queries

In [ ]:
# Additional test cases
test_queries = [
    "Compare the pricing and capabilities of Claude Sonnet 4 and Haiku 4, then recommend which to use for a simple classification task with 10 million tokens per month.",
    
    "What is the cost per query for a RAG system using Claude Sonnet 4 with typical token usage, and how does it compare to Haiku 4?",
    
    "Which AWS Bedrock model should I use for complex multi-hop RAG queries that require deep reasoning, and what would be the approximate cost for 100,000 queries?"
]

for idx, query in enumerate(test_queries[:1], 1):  # Test first query only to save time
    print(f"\n{'='*80}")
    print(f"TEST CASE {idx}")
    print(f"{'='*80}\n")
    
    result = agent.process_query(query)
    
    print(f"\nSub-Questions ({len(result['sub_questions'])}):")
    for i, sq in enumerate(result['sub_questions'], 1):
        print(f"  {i}. {sq}")
    
    print(f"\nFinal Answer:\n{result['final_answer']}")

## Performance Metrics

In [ ]:
def analyze_decomposition(result: Dict):
    """
    Analyze query decomposition performance.
    """
    metrics = {
        'num_sub_questions': len(result['sub_questions']),
        'tools_used': {},
        'avg_answer_length': 0
    }
    
    total_length = 0
    for sa in result['sub_answers']:
        tool = sa['tool']
        metrics['tools_used'][tool] = metrics['tools_used'].get(tool, 0) + 1
        total_length += len(sa['answer'].split())
    
    metrics['avg_answer_length'] = total_length / len(result['sub_answers']) if result['sub_answers'] else 0
    
    return metrics

metrics = analyze_decomposition(result)

print("\n" + "="*80)
print("PERFORMANCE METRICS")
print("="*80)
print(f"Sub-questions generated: {metrics['num_sub_questions']}")
print(f"Tools used: {dict(metrics['tools_used'])}")
print(f"Average answer length: {metrics['avg_answer_length']:.1f} words")
print(f"Final answer length: {len(result['final_answer'].split())} words")

## Key Benefits & Trade-offs

### Benefits
✅ **Handles Complex Queries**: Multi-hop reasoning across documents  
✅ **Transparent Reasoning**: Clear sub-questions show thought process  
✅ **Tool Orchestration**: Combines retrieval + computation  
✅ **Better Accuracy**: Breaks down ambiguity  

### Trade-offs
⚠️ **Higher Latency**: Sequential sub-question processing  
⚠️ **Increased Cost**: Multiple LLM calls (decomposition + sub-answers + synthesis)  
⚠️ **Error Propagation**: Mistakes in early sub-answers affect final result  
⚠️ **Complexity**: More moving parts than simple RAG  

### When to Use
- Complex analytical questions
- Comparison queries
- Multi-document synthesis
- Computational queries on retrieved data
- When transparency in reasoning is valued

### When NOT to Use
- Simple fact lookup
- Latency-critical applications (< 2s requirement)
- Cost-constrained scenarios
- Single-document queries

## Cleanup

In [ ]:
# Optional: Delete index
# os_client.indices.delete(index=index_name)
# print(f"Deleted index: {index_name}")